# Genetic Transformer

**A Transformer model bridging Natural Language and Genetic Sequences**

Inspired by *"Attention is All You Need"* (Vaswani et al., 2017), this notebook demonstrates a full encoder-decoder Transformer that translates bidirectionally between:
- Natural language descriptions ↔ DNA/RNA sequences
- Natural language queries ↔ Codon/amino acid/protein information

---

## Architecture Overview

```
┌─────────────────────────────────────────────────────────┐
│                   Genetic Transformer                    │
│                                                         │
│   ┌───────────┐    Cross-Attention    ┌───────────┐    │
│   │  Encoder   │ ─────────────────── │  Decoder   │    │
│   │  (4 layers)│                      │  (4 layers)│    │
│   │            │                      │            │    │
│   │ Self-Attn  │                      │ Masked     │    │
│   │ + FFN      │                      │ Self-Attn  │    │
│   │ + LayerNorm│                      │ + Cross    │    │
│   └─────┬──────┘                      │ + FFN      │    │
│         │                             └──────┬─────┘    │
│   ┌─────┴──────┐                      ┌──────┴─────┐   │
│   │ Positional  │                      │ Positional  │   │
│   │ Encoding    │                      │ Encoding    │   │
│   └─────┬──────┘                      └──────┬─────┘   │
│   ┌─────┴──────┐                      ┌──────┴─────┐   │
│   │  Shared     │◄────────────────────►│  Shared     │   │
│   │  Embedding  │   (unified vocab)    │  Embedding  │   │
│   └─────────────┘                      └─────────────┘   │
│                                                         │
│   Input: NL or Genetic Seq    Output: Genetic Seq or NL │
└─────────────────────────────────────────────────────────┘
```

## 1. Setup and Imports

In [ ]:
import sys
import os

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from genetic_transformer.config import GeneticTransformerConfig
from genetic_transformer.knowledge.codon_table import (
    CODON_TO_AMINO, AMINO_TO_CODONS, translate_rna_to_protein, translate_dna_to_rna
)
from genetic_transformer.knowledge.amino_acids import AMINO_ACID_PROPERTIES
from genetic_transformer.knowledge.constants import (
    DNA_BASES, RNA_BASES, START_CODONS, STOP_CODONS
)
from genetic_transformer.tokenizer import GeneticTokenizer, PAD_IDX
from genetic_transformer.model.transformer import GeneticTransformer
from genetic_transformer.data.synthetic_generator import SyntheticDataGenerator
from genetic_transformer.data.dataset import GeneticDataset, create_dataloaders
from genetic_transformer.training.trainer import Trainer
from genetic_transformer.inference.translator import GeneticTranslator
from genetic_transformer.inference.visualizer import AttentionVisualizer

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

## 2. Explore the Genetic Knowledge Base

Before building the model, let's explore the biological knowledge that powers it.

In [ ]:
# The Standard Genetic Code - All 64 Codons
print("=" * 70)
print("THE STANDARD GENETIC CODE")
print("=" * 70)
print(f"\nTotal codons: {len(CODON_TO_AMINO)}")
print(f"Start codons: {START_CODONS}")
print(f"Stop codons: {STOP_CODONS}")
print()

# Display codon table grouped by amino acid
print(f"{'Codon':<8} {'AA (1)':<8} {'AA (3)':<8} {'Full Name':<18} {'Type'}")
print("-" * 55)
for codon, info in list(CODON_TO_AMINO.items())[:20]:  # Show first 20
    print(f"{codon:<8} {info['one_letter']:<8} {info['three_letter']:<8} {info['full_name']:<18} {info['type']}")
print(f"... and {len(CODON_TO_AMINO) - 20} more codons")

In [ ]:
# Codon degeneracy - how many codons per amino acid
print("\nCodon Degeneracy (number of codons per amino acid):")
print("-" * 50)
for aa, codons in sorted(AMINO_TO_CODONS.items(), key=lambda x: len(x[1]), reverse=True):
    if aa == '*':
        name = 'Stop'
    else:
        name = AMINO_ACID_PROPERTIES[aa]['full_name']
    print(f"  {aa} ({name:<15s}): {len(codons)} codons -> {', '.join(codons)}")

In [ ]:
# Amino acid properties
print("\n" + "=" * 70)
print("AMINO ACID PROPERTIES")
print("=" * 70)
print(f"\n{'AA':<5} {'Name':<15} {'MW':<10} {'Charge':<10} {'Polarity':<12} {'Class'}")
print("-" * 70)
for code, props in AMINO_ACID_PROPERTIES.items():
    print(f"{code:<5} {props['full_name']:<15} {props['molecular_weight']:<10.1f} "
          f"{props['charge']:<10} {props['polarity']:<12} {props['classification']}")

In [ ]:
# Quick translation demo
print("\nTranslation Examples:")
print("-" * 40)

rna_seq = "AUGGCUUUUAAA"
protein = translate_rna_to_protein(rna_seq)
print(f"RNA: {rna_seq} -> Protein: {protein}")

dna_seq = "ATGGCTTTTAAA"
rna = translate_dna_to_rna(dna_seq)
print(f"DNA: {dna_seq} -> RNA: {rna}")

# Show what each codon means
print(f"\nCodon-by-codon breakdown of {rna_seq}:")
for i in range(0, len(rna_seq), 3):
    codon = rna_seq[i:i+3]
    info = CODON_TO_AMINO[codon]
    print(f"  {codon} -> {info['full_name']} ({info['one_letter']})")

## 3. Dual-Domain Tokenizer

Our tokenizer handles both natural language AND genetic sequences in a unified vocabulary.

In [ ]:
tokenizer = GeneticTokenizer()
print(f"Vocabulary size: {tokenizer.get_vocab_size()} tokens")
print(f"\nVocabulary breakdown:")
print(f"  Special tokens: <PAD>, <SOS>, <EOS>, <UNK>, <SEP>")
print(f"  Domain markers: <NL>, <DNA>, <RNA>, <PROTEIN>, <CODON>")
print(f"  Nucleotide bases: _A, _T, _G, _C, _U")
print(f"  Codons: 64 (all RNA codons as single tokens)")
print(f"  Amino acid tokens: 21 (<AA_M>, <AA_A>, ..., <AA_*>)")
print(f"  NL words: ~{tokenizer.get_vocab_size() - 64 - 21 - 10 - 5} biology + common English words")

In [ ]:
# Tokenization examples
examples = [
    ("what amino acid does the codon AUG encode", "auto"),
    ("AUGCCUUAA", "auto"),
    ("ATGCCT", "auto"),
    ("M A F", "protein"),
    ("the start codon is AUG which encodes methionine", "nl"),
]

print("Tokenization Examples:")
print("=" * 70)
for text, mode in examples:
    detected = tokenizer.detect_domain(text)
    tokens = tokenizer.tokenize(text, mode)
    print(f"\nInput:    '{text}'")
    print(f"Detected: {detected}")
    print(f"Tokens:   {tokens}")
    
    # Encode and decode round-trip
    encoded = tokenizer.encode(text, mode, max_len=32)
    decoded = tokenizer.decode(encoded)
    print(f"Encoded:  {encoded[:15]}... (len={sum(1 for x in encoded if x != 0)})")
    print(f"Decoded:  '{decoded}'")

## 4. Build and Inspect the Model

In [ ]:
# Configuration
config = GeneticTransformerConfig(
    d_model=256,
    n_heads=4,
    n_encoder_layers=4,
    n_decoder_layers=4,
    d_ff=512,
    dropout=0.1,
    max_seq_len=128,
    batch_size=64,
    num_epochs=20,
    warmup_steps=400,
    num_train_samples=30000,
    num_val_samples=3000,
)

# Set vocab sizes from tokenizer
vocab_size = tokenizer.get_vocab_size()
config.src_vocab_size = vocab_size
config.tgt_vocab_size = vocab_size

print(f"Configuration:")
print(f"  d_model:      {config.d_model}")
print(f"  n_heads:      {config.n_heads}")
print(f"  n_layers:     {config.n_encoder_layers} (encoder) / {config.n_decoder_layers} (decoder)")
print(f"  d_ff:         {config.d_ff}")
print(f"  vocab_size:   {vocab_size}")
print(f"  max_seq_len:  {config.max_seq_len}")

In [ ]:
# Build the model
model = GeneticTransformer(config).to(device)

n_params = model.count_parameters()
print(f"\nModel built successfully!")
print(f"Total trainable parameters: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"\nModel architecture:")
print(model)

In [ ]:
# Verify forward pass with random data
print("\nForward pass verification:")
dummy_src = torch.randint(0, vocab_size, (2, 32), device=device)
dummy_tgt = torch.randint(0, vocab_size, (2, 32), device=device)

logits, attn_maps = model(dummy_src, dummy_tgt)
print(f"  Input shapes:  src={dummy_src.shape}, tgt={dummy_tgt.shape}")
print(f"  Output logits: {logits.shape}  (batch, seq_len, vocab_size)")
print(f"  Attention maps:")
for key, val in attn_maps.items():
    if isinstance(val, list):
        print(f"    {key}: {len(val)} layers, each {val[0].shape}")
    else:
        print(f"    {key}: {val.shape}")
print("  \u2713 Forward pass successful!")

## 5. Generate Synthetic Training Data

We generate paired (source, target) examples from the codon table and biological rules.

In [ ]:
# Generate and inspect training data
generator = SyntheticDataGenerator(seed=42)
sample_pairs = generator.generate_all(100)

print("Sample Training Pairs:")
print("=" * 80)
for i, (src, tgt) in enumerate(sample_pairs[:15]):
    print(f"\n[{i+1}] Source: {src}")
    print(f"    Target: {tgt}")

In [ ]:
# Create full training/validation dataloaders
print(f"Generating {config.num_train_samples:,} training + {config.num_val_samples:,} validation pairs...")
train_loader, val_loader = create_dataloaders(config, tokenizer)

print(f"\nDataLoaders created:")
print(f"  Train: {len(train_loader)} batches (batch_size={config.batch_size})")
print(f"  Val:   {len(val_loader)} batches")

# Inspect one batch
batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  src:   {batch['src'].shape}")
print(f"  tgt:   {batch['tgt'].shape}")
print(f"  tgt_y: {batch['tgt_y'].shape}")

## 6. Training

Train the Genetic Transformer using label-smoothed cross-entropy and the warmup+decay LR schedule from the original paper.

In [ ]:
# Re-initialize model for clean training
model = GeneticTransformer(config).to(device)
print(f"Model parameters: {model.count_parameters():,}")
print(f"Training for {config.num_epochs} epochs on {device}...")
print()

trainer = Trainer(model, config, train_loader, val_loader, device=device)
history = trainer.train(verbose=True)

In [ ]:
# Plot training curves
fig = AttentionVisualizer.plot_training_curves(history)
plt.show()

## 7. Interactive Translation

Let's test the trained model with various queries in both directions.

In [ ]:
# Create translator
translator = GeneticTranslator(model, tokenizer, config, device=device)

print("=" * 70)
print("NATURAL LANGUAGE -> GENETIC TRANSLATION")
print("=" * 70)

nl_queries = [
    "what amino acid does the codon AUG encode",
    "what amino acid does the codon GCU encode",
    "translate the rna sequence AUG GCU UUU UAA",
    "transcribe ATGCCC to rna",
    "what are the stop codons",
    "is methionine polar",
    "what is the start codon",
]

for query in nl_queries:
    result = translator.translate(query)
    print(f"\n  Q: {query}")
    print(f"  A: {result}")

In [ ]:
print("\n" + "=" * 70)
print("GENETIC SEQUENCE -> NATURAL LANGUAGE")
print("=" * 70)

genetic_queries = [
    "AUG GCU UAA",
    "AUG UUU GCU AAA",
    "AUG CCU GGU UGG",
]

for query in genetic_queries:
    result = translator.translate(query)
    # Also show the programmatic translation for comparison
    protein = translate_rna_to_protein(query.replace(' ', ''))
    print(f"\n  Input:    {query}")
    print(f"  Model:    {result}")
    print(f"  Expected: Protein = {protein}")

In [ ]:
# Try your own queries!
print("\n" + "=" * 70)
print("TRY YOUR OWN QUERIES")
print("=" * 70)

# Edit these to try different inputs:
custom_queries = [
    "what amino acid does UGG encode",
    "translate AUG GAA UGC UAG",
    "is alanine hydrophobic",
    "what is the reverse complement of ATGCCA",
]

for query in custom_queries:
    result = translator.translate(query)
    print(f"\n  Q: {query}")
    print(f"  A: {result}")

## 8. Attention Visualization

Visualize what the model attends to when translating. Cross-attention maps show which source tokens the decoder focuses on when generating each output token.

In [ ]:
# Translate with attention maps
test_input = "what amino acid does the codon AUG encode"
output, attn_maps, src_tokens, tgt_tokens = translator.translate_with_attention(test_input)

print(f"Input:  {test_input}")
print(f"Output: {output}")
print(f"Source tokens: {src_tokens}")
print(f"Target tokens: {tgt_tokens}")

In [ ]:
# Cross-attention heatmap (averaged across heads)
fig = AttentionVisualizer.plot_cross_attention(attn_maps, src_tokens, tgt_tokens)
plt.show()

In [ ]:
# Multi-head attention for the last decoder layer
cross_attn = attn_maps.get('decoder_cross', attn_maps.get('decoder_cross_attn', []))
if cross_attn:
    fig = AttentionVisualizer.plot_multi_head_attention(
        cross_attn[-1], src_tokens, tgt_tokens, 
        layer=len(cross_attn)-1, n_heads=config.n_heads
    )
    plt.suptitle('Decoder Cross-Attention (Last Layer) - All Heads', fontsize=14)
    plt.show()

In [ ]:
# Attention on a genetic sequence input
test_genetic = "AUG GCU UUU UAA"
output2, attn_maps2, src_tokens2, tgt_tokens2 = translator.translate_with_attention(test_genetic)

print(f"Input:  {test_genetic}")
print(f"Output: {output2}")

fig = AttentionVisualizer.plot_cross_attention(attn_maps2, src_tokens2, tgt_tokens2)
plt.show()

In [ ]:
# Encoder self-attention (how source tokens relate to each other)
enc_self_attn = attn_maps.get('encoder_self', attn_maps.get('encoder_self_attn', []))
if enc_self_attn:
    fig = AttentionVisualizer.plot_attention_heatmap(
        enc_self_attn[-1], src_tokens, src_tokens,
        title='Encoder Self-Attention (Last Layer, Head 0)',
        layer=len(enc_self_attn)-1, head=0
    )
    plt.show()

## Summary

This Genetic Transformer demonstrates:

1. **Custom dual-domain tokenizer** handling both NL and genetic sequences in a unified vocabulary
2. **Full encoder-decoder Transformer** from the original 2017 paper with multi-head attention
3. **Bidirectional translation** between natural language and DNA/RNA/protein sequences
4. **Synthetic data generation** from the standard genetic code and biological rules
5. **Attention visualization** showing how the model attends to source tokens during translation

### Key architectural choices:
- **Shared embedding** between encoder and decoder (unified vocab)
- **Codons as atomic tokens** (preserving biological meaning)
- **Sinusoidal positional encoding** (from the original paper)
- **Label smoothing** to prevent overconfidence on synthetic data
- **Warmup + inverse-sqrt LR schedule** (from the original paper)